# Amazon India Sales Analytics — Validated Workflow

This notebook reproduces the cleaning and analytical summaries from the supplied **Amazon Sale Report.csv**.

## Run instructions
1. Put the original source file named `Amazon Sale Report.csv` beside this notebook (the raw order-level file is intentionally not committed to the public repository).
2. Install dependencies with `pip install -r requirements.txt`.
3. Run all cells from top to bottom.

## Data handling and interpretation
- The source includes both `MM-DD-YY` and `MM-DD-YYYY` date strings; both are parsed explicitly.
- Three export/helper columns are removed: `index`, `New`, and `PendingS`.
- Exact duplicate rows are removed.
- Missing `Amount` values are not imputed.
- `Amount` is recorded sales, not profit.
- The public dashboard uses aggregated outputs; raw order identifiers and postal codes are not published.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import hashlib

SOURCE = Path("Amazon Sale Report.csv")
if not SOURCE.exists():
    raise FileNotFoundError("Place the original 'Amazon Sale Report.csv' beside this notebook before running.")

raw = pd.read_csv(SOURCE, low_memory=False)
print("Raw shape:", raw.shape)
display(raw.head())


In [ ]:
# Cleaning: remove export/helper columns, then exact duplicate rows.
helper_columns = ["index", "New", "PendingS"]
missing_helper_columns = [c for c in helper_columns if c not in raw.columns]
if missing_helper_columns:
    raise KeyError(f"Expected source columns missing: {missing_helper_columns}")

df = raw.drop(columns=helper_columns).drop_duplicates().copy()
rows_removed = len(raw) - len(df)

# Parse both date formats found in the source.
date_short_year = pd.to_datetime(df["Date"], format="%m-%d-%y", errors="coerce")
date_long_year = pd.to_datetime(df["Date"], format="%m-%d-%Y", errors="coerce")
df["Date"] = date_short_year.fillna(date_long_year)

# Preserve postal codes as strings for local analysis; do not publish them.
df["ship-postal-code"] = df["ship-postal-code"].astype("string")

print("Cleaned shape:", df.shape)
print("Exact duplicate rows removed:", rows_removed)
print("Invalid dates after parsing:", int(df["Date"].isna().sum()))
print("Missing Amount:", int(df["Amount"].isna().sum()))
assert df.duplicated().sum() == 0
assert df["Date"].notna().all(), "Unexpected date parsing failures; inspect source values."


In [ ]:
# Data-quality profile
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2)
})
display(quality.sort_values("missing_count", ascending=False))
print("Rows:", len(df), "| Distinct orders:", df["Order ID"].nunique())


In [ ]:
# Core KPIs. AOV = mean of recorded order-level Amount totals.
order_totals = df.groupby("Order ID")["Amount"].sum(min_count=1)
total_sales = df["Amount"].sum(min_count=1)
unique_orders = df["Order ID"].nunique()
aov = order_totals.mean()

kpis = pd.DataFrame([{
    "recorded_sales_inr": total_sales,
    "distinct_orders": unique_orders,
    "units": df["Qty"].sum(),
    "aov_inr": aov,
    "missing_amount_rows": int(df["Amount"].isna().sum()),
    "cancelled_orders": df.loc[df["Status"].eq("Cancelled"), "Order ID"].nunique(),
}])
display(kpis)


In [ ]:
# Python aggregations
monthly = (df.dropna(subset=["Date"])
    .assign(Sales_Month=lambda x: x["Date"].dt.to_period("M").astype(str))
    .groupby("Sales_Month", as_index=False)
    .agg(Total_Sales=("Amount", "sum"),
         Total_Orders=("Order ID", "nunique"),
         Total_Quantity=("Qty", "sum")))

category = (df.groupby("Category", as_index=False)
    .agg(Total_Sales=("Amount", "sum"),
         Total_Quantity=("Qty", "sum"),
         Total_Orders=("Order ID", "nunique"))
    .sort_values("Total_Sales", ascending=False))

states = (df.dropna(subset=["ship-state"])
    .groupby("ship-state", as_index=False)
    .agg(Total_Sales=("Amount", "sum"), Total_Orders=("Order ID", "nunique"))
    .sort_values("Total_Sales", ascending=False).head(10))

status = (df.groupby("Status", as_index=False, dropna=False)
    .agg(Total_Orders=("Order ID", "nunique"),
         Recorded_Sales=("Amount", lambda s: s.sum(min_count=1)),
         Total_Quantity=("Qty", "sum"))
    .sort_values("Total_Orders", ascending=False))

segment = (df.assign(Customer_Segment=np.where(df["B2B"].astype(bool), "B2B", "B2C"))
    .groupby("Customer_Segment", as_index=False)
    .agg(Total_Sales=("Amount", "sum"),
         Total_Orders=("Order ID", "nunique"),
         Total_Quantity=("Qty", "sum")))

display(monthly, category, states, status, segment)


In [ ]:
# DuckDB SQL analysis
con = duckdb.connect(database=":memory:")
con.register("amazon_sales", df)

kpi_sql = con.execute("""
SELECT SUM(Amount) AS Total_Sales,
       COUNT(DISTINCT "Order ID") AS Total_Orders,
       SUM(Qty) AS Total_Quantity,
       ROUND(SUM(Amount) / NULLIF(COUNT(DISTINCT "Order ID"), 0), 2) AS Average_Order_Value
FROM amazon_sales
""").df()

monthly_sql = con.execute("""
SELECT strftime(Date, '%Y-%m') AS Sales_Month,
       ROUND(SUM(Amount), 2) AS Total_Sales,
       COUNT(DISTINCT "Order ID") AS Total_Orders,
       SUM(Qty) AS Total_Quantity
FROM amazon_sales
WHERE Date IS NOT NULL
GROUP BY 1 ORDER BY 1
""").df()

category_sql = con.execute("""
SELECT Category, ROUND(SUM(Amount), 2) AS Total_Sales,
       SUM(Qty) AS Total_Quantity, COUNT(DISTINCT "Order ID") AS Total_Orders
FROM amazon_sales GROUP BY Category ORDER BY Total_Sales DESC
""").df()

state_sql = con.execute("""
SELECT "ship-state" AS State, ROUND(SUM(Amount), 2) AS Total_Sales,
       COUNT(DISTINCT "Order ID") AS Total_Orders
FROM amazon_sales WHERE "ship-state" IS NOT NULL
GROUP BY State ORDER BY Total_Sales DESC LIMIT 10
""").df()

status_sql = con.execute("""
SELECT Status, COUNT(DISTINCT "Order ID") AS Total_Orders,
       ROUND(SUM(Amount), 2) AS Recorded_Sales, SUM(Qty) AS Total_Quantity
FROM amazon_sales GROUP BY Status ORDER BY Total_Orders DESC
""").df()

segment_sql = con.execute("""
SELECT CASE WHEN B2B THEN 'B2B' ELSE 'B2C' END AS Customer_Segment,
       ROUND(SUM(Amount), 2) AS Total_Sales,
       COUNT(DISTINCT "Order ID") AS Total_Orders, SUM(Qty) AS Total_Quantity
FROM amazon_sales GROUP BY B2B ORDER BY Total_Sales DESC
""").df()

display(kpi_sql, monthly_sql, category_sql, state_sql, status_sql, segment_sql)


In [ ]:
# Reconciliation assertions: DuckDB results must match Pandas.
assert round(float(kpi_sql.loc[0, "Total_Sales"]), 2) == round(float(total_sales), 2)
assert int(kpi_sql.loc[0, "Total_Orders"]) == int(unique_orders)
assert int(kpi_sql.loc[0, "Total_Quantity"]) == int(df["Qty"].sum())
assert round(float(kpi_sql.loc[0, "Average_Order_Value"]), 2) == round(float(aov), 2)

assert monthly_sql["Sales_Month"].tolist() == monthly["Sales_Month"].tolist()
assert np.allclose(monthly_sql["Total_Sales"], monthly["Total_Sales"], atol=0.01)
assert category_sql["Category"].tolist() == category["Category"].tolist()
assert np.allclose(category_sql["Total_Sales"], category["Total_Sales"], atol=0.01)
assert state_sql["State"].tolist() == states["ship-state"].tolist()
assert np.allclose(state_sql["Total_Sales"], states["Total_Sales"], atol=0.01)
assert status_sql["Status"].tolist() == status["Status"].tolist()
assert np.allclose(status_sql["Total_Orders"], status["Total_Orders"])
assert np.allclose(status_sql["Recorded_Sales"], status["Recorded_Sales"], equal_nan=True)
assert segment_sql["Customer_Segment"].tolist() == segment.sort_values("Total_Sales", ascending=False)["Customer_Segment"].tolist()
assert np.allclose(segment_sql["Total_Sales"], segment.sort_values("Total_Sales", ascending=False)["Total_Sales"], atol=0.01)

print("PASS — DuckDB and Pandas reconcile for KPIs, monthly, category, state, status and B2B/B2C.")


In [ ]:
# Public-safe local export: replace order IDs with stable pseudonymous keys and remove postal codes.
public_df = df.copy()
public_df["Order_Key"] = public_df["Order ID"].astype(str).map(
    lambda value: hashlib.sha256(value.encode("utf-8")).hexdigest()[:16]
)
public_df = public_df.drop(columns=["Order ID", "ship-postal-code"])
public_df.to_csv("Amazon_Sales_Cleaned_Public.csv", index=False)

# Export aggregate tables used by the dashboard/report.
monthly.to_csv("Amazon_Sales_Monthly.csv", index=False)
category.to_csv("Amazon_Sales_Category.csv", index=False)
states.to_csv("Amazon_Sales_Top_States.csv", index=False)
status.to_csv("Amazon_Sales_Status.csv", index=False)
segment.to_csv("Amazon_Sales_Segment.csv", index=False)

print("Public-safe and aggregate outputs exported. Raw identifiers and postal codes are excluded.")


## Verified findings and caveats

The corrected source rebuild produced **128,017 rows** from **128,976 original rows**, removing **959 exact duplicate rows**. Mixed-format date parsing succeeded for all 128,017 retained rows. There are **7,563 missing Amount values (5.91%)**; these remain missing and are not imputed.

The validated descriptive totals are ₹78,179,803.88 recorded Amount, 120,229 distinct order IDs, and ₹650.26 average recorded order value. The AOV is defined as the mean of order-level Amount totals.

March 2022 is partial (only 31 March records), so April month-on-month growth is not a full-month like-for-like comparison. The dataset does not support profit/margin, customer retention/CLV, or delivery-time analysis because required cost, customer, and timestamp fields are unavailable.

**Important:** the raw source file is not committed to the public repository. The notebook can be reproduced by a reviewer who has lawful access to that source file.